# Q-Learning 기초

Q-Learning 알고리즘을 이해하고 간단한 환경에서 구현합니다.

## 학습 목표
1. 강화학습 기본 개념
2. Q-Learning 알고리즘 이해
3. OpenAI Gym 사용법
4. 탐험-활용 균형

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from tqdm import tqdm

np.random.seed(42)

## 1. 강화학습 기본 개념

- **상태 (State)**: 환경의 현재 상황
- **행동 (Action)**: 에이전트가 취할 수 있는 동작
- **보상 (Reward)**: 행동에 대한 피드백
- **정책 (Policy)**: 상태에서 행동을 선택하는 전략

In [ ]:
# FrozenLake 환경
env = gym.make('FrozenLake-v1', is_slippery=False, render_mode='ansi')

print(f"상태 공간: {env.observation_space.n}")
print(f"행동 공간: {env.action_space.n}")
print("\n행동:")
print("0: 왼쪽, 1: 아래, 2: 오른쪽, 3: 위")

# 환경 시각화
state, info = env.reset()
print("\n환경:")
print(env.render())

## 2. Q-Learning 알고리즘

$$Q(s, a) \leftarrow Q(s, a) + \alpha [r + \gamma \max_{a'} Q(s', a') - Q(s, a)]$$

In [ ]:
class QLearningAgent:
    def __init__(self, state_size, action_size, learning_rate=0.1, 
                 discount_factor=0.99, epsilon=1.0, epsilon_decay=0.995, 
                 epsilon_min=0.01):
        self.state_size = state_size
        self.action_size = action_size
        self.lr = learning_rate
        self.gamma = discount_factor
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        
        # Q-테이블 초기화
        self.q_table = np.zeros((state_size, action_size))
    
    def choose_action(self, state):
        """Epsilon-greedy 정책으로 행동 선택"""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.action_size)  # 탐험
        return np.argmax(self.q_table[state])  # 활용
    
    def learn(self, state, action, reward, next_state, done):
        """Q-값 업데이트"""
        current_q = self.q_table[state, action]
        
        if done:
            target_q = reward
        else:
            target_q = reward + self.gamma * np.max(self.q_table[next_state])
        
        # Q-Learning 업데이트
        self.q_table[state, action] += self.lr * (target_q - current_q)
    
    def decay_epsilon(self):
        """Epsilon 감소"""
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

In [ ]:
# 에이전트 생성
agent = QLearningAgent(
    state_size=env.observation_space.n,
    action_size=env.action_space.n
)

# 학습
episodes = 1000
rewards_per_episode = []

for episode in tqdm(range(episodes)):
    state, info = env.reset()
    total_reward = 0
    done = False
    
    while not done:
        action = agent.choose_action(state)
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        agent.learn(state, action, reward, next_state, done)
        
        state = next_state
        total_reward += reward
    
    agent.decay_epsilon()
    rewards_per_episode.append(total_reward)

print(f"\n학습 완료!")
print(f"최종 Epsilon: {agent.epsilon:.4f}")
print(f"최근 100 에피소드 평균 보상: {np.mean(rewards_per_episode[-100:]):.2f}")

In [ ]:
# 학습 곡선
window_size = 50
moving_avg = np.convolve(rewards_per_episode, np.ones(window_size)/window_size, mode='valid')

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(rewards_per_episode, alpha=0.3)
plt.plot(moving_avg, color='red')
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.title('Training Progress')

plt.subplot(1, 2, 2)
plt.imshow(agent.q_table, cmap='hot', aspect='auto')
plt.colorbar()
plt.xlabel('Action')
plt.ylabel('State')
plt.title('Q-Table')

plt.tight_layout()
plt.show()

In [ ]:
# 학습된 정책 테스트
def test_agent(agent, env, episodes=10):
    success_count = 0
    
    for _ in range(episodes):
        state, info = env.reset()
        done = False
        
        while not done:
            action = np.argmax(agent.q_table[state])  # Greedy
            state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            
            if reward == 1:
                success_count += 1
    
    return success_count / episodes

success_rate = test_agent(agent, env, episodes=100)
print(f"성공률: {success_rate:.1%}")

## 3. 학습된 정책 시각화

In [ ]:
# 최적 정책 시각화
policy = np.argmax(agent.q_table, axis=1).reshape(4, 4)
action_symbols = {0: '←', 1: '↓', 2: '→', 3: '↑'}

print("학습된 정책:")
for i in range(4):
    row = ""
    for j in range(4):
        row += action_symbols[policy[i, j]] + " "
    print(row)

print("\nFrozenLake 맵:")
print("S: 시작, F: 얼음, H: 구멍, G: 목표")

## 연습 문제

1. is_slippery=True로 설정하고 학습해보세요.
2. 하이퍼파라미터 (learning_rate, discount_factor)를 조정해보세요.
3. Taxi-v3 환경에서 Q-Learning을 구현해보세요.
4. SARSA 알고리즘을 구현하고 비교해보세요.